# Run the FULFILL sufficiency model

This notebook solves the **full 43-region hybrid SUT**, for a chosen case and its **same-year/background no-measure control**. It may download about 129.6 MB of inputs on first use, and needs several GB of memory. Allow several minutes for two dense solves.

The default selects NZE 2050, all six measures. To try your own combination, change CASE_FILE to `config/diet-and-aviation.json`. Input preparation and the historical study notebooks remain available separately.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from fulfill_model import Case, build_plan
from fulfill_model.data import ensure_data

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'Data'

import json
import warnings
from fulfill_model.runtime import load_database, run_plan, compare_accounts
warnings.filterwarnings('ignore', message='indexing past lexsort depth.*', category=pd.errors.PerformanceWarning)


## Select a case and the account variant

The prepared public ZIP omits the extra aviation account created in the paper's first notebook. True explicitly adds that account using the same formula. False preserves the public ZIP's account set. The variant is recorded in run.json.

In [2]:
CASE_FILE = ROOT / 'config/paper-all.json'
ADD_AVIATION_ACCOUNT = True
OFFLINE = False
OUTPUT = ROOT / 'Outputs/runs/notebook-comparison'

case = Case.from_json(CASE_FILE)
control_case = Case.paper(case.background, '0', case.year)
if case.identifier == control_case.identifier:
    raise ValueError('Select a measure case for this paired comparison.')
if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise FileExistsError('Choose a new OUTPUT folder for a new calculation.')
display(pd.Series(case.to_dict(), name='Selected case'))

name                NZE_All_2050
background                   NZE
year                        2050
measures      [D, S, M, P, F, B]
adoption                      {}
Name: Selected case, dtype: object

## Download checked inputs and inspect the selected blocks

In [3]:
ensure_data(DATA, offline=OFFLINE, progress=lambda _: None)
control_plan = build_plan(control_case, inputs=DATA / 'Files for shocks')
scenario_plan = build_plan(case, inputs=DATA / 'Files for shocks')
display(scenario_plan.steps[['step','matrix','archetype','measure','adoption']])

,step,matrix,archetype,measure,adoption
0,cons_var,Y,scale_final_demand,background,1.0
1,car-eff,z,technical_requirement,M,1.0
2,hs-eff,z,technical_requirement,background,1.0
3,car-mix,z,supply_mix,background,1.0
4,hous,z,technical_requirement,background,1.0
5,hs-mix,z,supply_mix,background,1.0
6,ele-mix,z,electricity_mix,background,1.0
7,m2pc,Y,per_capita_demand,S,1.0
8,meals_dem,Y,total_service_demand,background,1.0
9,mvkm,Y,per_capita_demand,B,1.0


## Load the prepared model

The original 2011 baseline is retained. Each case applies its own background/year instructions to that baseline; scenarios are not applied cumulatively.

In [4]:
print('Loading the full published database...')
db = load_database(DATA, air_transport_extra=ADD_AVIATION_ACCOUNT, progress=lambda _: None)
print('Regions:', len(db.get_index('Region')))
print('Extra aviation account:', 'Air transport extra emissions' in db.E.index)

Loading the full published database...


Using add extensions will rewrite the new results on the baseline and delete other scenarios


baseline deleted from the database


Database: to calculate z following matrices are need.
['X'].Trying to calculate dependencies.


Regions: 43
Extra aviation account: True


## Calculate the control and selected scenario

Each solve exports the ordered instructions, original-unit accounts and production quantities. Solved scenarios are released from memory after export.

In [5]:
print('Calculating the same-year background control...')
control = run_plan(db, control_plan, OUTPUT / control_case.identifier, progress=lambda _: None)
print('Calculating the selected sufficiency case...')
scenario = run_plan(db, scenario_plan, OUTPUT / case.identifier, progress=lambda _: None)
comparison = compare_accounts(control, scenario)
comparison.to_csv(OUTPUT / 'comparison.csv', index=False)
timings = [dict(case=p.case.identifier, solve_seconds=json.loads((OUTPUT/p.case.identifier/'run.json').read_text())['solve_seconds'])
           for p in (control_plan, scenario_plan)]
display(pd.DataFrame(timings))

Calculating the same-year background control...


Calculating the selected sufficiency case...


,case,solve_seconds
0,NZE_0_2050,122.835273
1,NZE_All_2050,115.104516


## Inspect results with an explicit boundary

The table shows **fossil CO2 in production**, by producing region, in tonnes. It is not an EU consumption footprint or a complete reproduction of the paper figures. Other emission, resource, employment and value-added accounts remain in their original units.

In [6]:
selected = comparison[(comparison.matrix == 'E') &
                      (comparison.account == 'Carbon dioxide, fossil') &
                      comparison.region.isin(['DE','FR','IT','DK','LV'])]
display(selected[['region','unit','control_value','scenario_value','difference','relative_difference']])
print('Saved comparison and full account tables; the run manifests record sources, code, variant and timing.')

,region,unit,control_value,scenario_value,difference,relative_difference
182,DE,tonnes,7.432621e+08,6.913409e+08,-5.192115e+07,-0.069856
183,DK,tonnes,9.663429e+07,9.199303e+07,-4.641261e+06,-0.048029
187,FR,tonnes,3.596422e+08,3.313692e+08,-2.827296e+07,-0.078614
195,IT,tonnes,3.198244e+08,2.965769e+08,-2.324747e+07,-0.072688
200,LV,tonnes,1.722234e+07,1.554355e+07,-1.678789e+06,-0.097477


Saved comparison and full account tables; the run manifests record sources, code, variant and timing.


## Further exploration

Change the JSON case, use a new output folder and rerun. Do not multiply the calculated impacts by the adoption parameter: solve the scenario with its adjusted assumptions. Use `fulfill reproduce` for the exact historical XLSX files, especially the 24 BAU cases documented in the verification guide.